In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [11]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This...
  """
  # Convert the input to small/lower order.
  data = text.lower()
  # Remove URLs
  data = re.sub(r'http\S+|www\S+', '', data)
  # Remove emojis
  data = re.sub(r'@\w+', '', data)
  # Remove all other unwanted characters.
  data = data.encode('ascii', 'ignore').decode('ascii')
  # Create tokens.
  tokens = data.split()
  # Remove stopwords:
  tokens = data.split()
  if rule == "lemmatize":
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")


  return " ".join(tokens)


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [12]:
df = pd.read_csv('/content/drive/MyDrive/COLLEGE/Colab Notebooks/AI AND MACHINE LEARNING/week8/trum_tweet_sentiment_analysis.csv')

print(df.head())

                                                text  Sentiment
0  RT @JohnLeguizamo: #trump not draining swamp b...          0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2  Trump protests: LGBTQ rally in New York https:...          1
3  "Hi I'm Piers Morgan. David Beckham is awful b...          0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0


Apply Cleaning

In [15]:
def text_cleaning_pipeline(text_input, rule = "lemmatize"):
  """
  This function cleans and preprocesses text data by lowercasing, removing URLs, emojis, and other unwanted characters. It also tokenizes the text, removes stopwords, and can apply lemmatization or stemming.
  """
  # Convert the input to small/lower order.
  data = text_input.lower()
  # Remove URLs
  data = re.sub(r'http\S+|www\S+', '', data)
  # Remove mentions (previously emojis, but pattern @\w+ targets mentions)
  data = re.sub(r'@\w+', '', data)
  # Remove other special characters and punctuation, keeping only letters and spaces
  data = re.sub(r'[^a-z\s]', '', data)
  # Remove all other unwanted characters (non-ascii)
  data = data.encode('ascii', 'ignore').decode('ascii')

  # Create tokens.
  tokens = data.split()

  # Initialize lemmatizer and stopwords
  lemmatizer = WordNetLemmatizer()
  stop_words = set(stopwords.words('english'))

  # Remove stopwords:
  tokens = [word for word in tokens if word not in stop_words and len(word) > 1]

  if rule == "lemmatize":
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    # Stemmer needs to be initialized if used
    stemmer = nltk.stem.PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)


                                                text  \
0  RT @JohnLeguizamo: #trump not draining swamp b...   
1  ICYMI: Hackers Rig FM Radio Stations To Play A...   
2  Trump protests: LGBTQ rally in New York https:...   
3  "Hi I'm Piers Morgan. David Beckham is awful b...   
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...   

                                          clean_text  
0  rt trump draining swamp taxpayer dollar trip a...  
1  icymi hacker rig fm radio station play antitru...  
2    trump protest lgbtq rally new york bbcworld via  
3  hi im pier morgan david beckham awful donald t...  
4  rt tech firm suing buzzfeed publishing unverif...  


In [16]:
df['clean_text'] = df['text'].apply(text_cleaning_pipeline)

print(df[['text', 'clean_text']].head())

                                                text  \
0  RT @JohnLeguizamo: #trump not draining swamp b...   
1  ICYMI: Hackers Rig FM Radio Stations To Play A...   
2  Trump protests: LGBTQ rally in New York https:...   
3  "Hi I'm Piers Morgan. David Beckham is awful b...   
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...   

                                          clean_text  
0  rt trump draining swamp taxpayer dollar trip a...  
1  icymi hacker rig fm radio station play antitru...  
2    trump protest lgbtq rally new york bbcworld via  
3  hi im pier morgan david beckham awful donald t...  
4  rt tech firm suing buzzfeed publishing unverif...  


Train-Test Split

In [18]:
X = df['clean_text']
y = df['Sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

TF-IDF Vectorization

In [19]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

Model Training (Logistic Regression)

In [20]:
model = LogisticRegression()

model.fit(X_train_tfidf, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

Prediction & Evaluation

In [21]:
y_pred = model.predict(X_test_tfidf)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      0.97      0.96    248563
           1       0.94      0.91      0.92    121462

    accuracy                           0.95    370025
   macro avg       0.95      0.94      0.94    370025
weighted avg       0.95      0.95      0.95    370025

